## Import Libraries

In [ ]:
import numpy as np
from qiskit import QuantumCircuit,transpile, ClassicalRegister, QuantumRegister
from qiskit.quantum_info import Kraus, SuperOp
from qiskit.visualization import plot_histogram
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import Statevector,DensityMatrix,state_fidelity,partial_trace, Operator
from matplotlib import pyplot as plt
from functools import reduce
from scipy.linalg import expm
import pandas as pd
import qiskit.circuit.classical as qiskit_classical
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2
from qiskit_aer import AerSimulator
from IPython.display import display
#from qiskit.opflow import I, Z, StateFn, PauliExpectation, CircuitSampler
#from qiskit import Aer, execute, transpile
from qiskit_ibm_runtime import SamplerV2
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke, FakeBrisbane
# Import from Qiskit Aer noise module
from qiskit_aer.noise import (
    NoiseModel,
    QuantumError,
    ReadoutError,
    depolarizing_error,
    pauli_error,
    thermal_relaxation_error,
)

Hamiltonian:

$H=-h(X_1+X_2)-J Z_1 Z_2$

Trotter formula:

$e^{-iH t}=(e^{i h X_1 \Delta t}e^{i h X_2 \Delta t}e^{i J Z_1 Z_2 \Delta t})^{N_{\text{Trot}}},$

where $\Delta t=t/N_{\text{Trot}}$.

## AER Pure SWAPNET

### Set SWAPNET

In [13]:
service = QiskitRuntimeService()
fake_backend = FakeSherbrooke()

/var/folders/hv/vrlxqgrs1bl41tsgjk0ptb280000gn/T/ipykernel_11424/1190649648.py:1: DeprecationWarning: The "ibm_quantum" channel option is deprecated and will be sunset on 1 July. After this date, ibm_cloud will be the only valid channel. For information on migrating to the new IBM Quantum Platform on the "ibm_cloud" channel, review the migration guide https://quantum.cloud.ibm.com/docs/migration-guides/classic-iqp-to-cloud-iqp .
  service = QiskitRuntimeService()


In [35]:
class ising_class:
    def __init__(self, k, steps, t, J, h):
        self.k = k
        self.steps = steps
        self.t = t
        self.J = J
        self.h = h

    def get_trotterized_ising_circuit(self):
        
        """
        Returns a QuantumCircuit implementing a trotterized Ising evolution for k qubits.

        H = - J * sum(Z_i Z_{i+1}) - h * sum(X_i)
        U = exp(-i H t) 
        """
        t = self.t
        steps = self.steps
        k = self.k
        J = self.J
        h = self.h

        dt = t / steps
        qc = QuantumCircuit(k)

        for _ in range(steps):
            # Apply ZZ interactions (Z_i Z_{i+1})
            for i in range(k - 1):
                qc.cx(i, i + 1)
                qc.rz(-2 * J * dt, i + 1)
                qc.cx(i, i + 1)

            # Apply transverse field X terms (X_i)
            for i in range(k):
                qc.rx(-2 * h * dt, i)

        return qc

    def apply_ising_to_registers(self, qc,start):
        """
        Apply trotterized Ising circuit to registers q1, q2, q3 in a 4d-sized register circuit.
        """
        k = self.k
        ising = self.get_trotterized_ising_circuit()

        # Convert to instruction and append to registers q1, q2, q3
        ising_inst = ising.to_instruction()
        for reg in [1, 2, 3]:
            qc.append(ising_inst, [(reg-1) * k + i + start for i in range(k)])

        return qc

    def get_trotterized_ising_statevector(self):
        """
        Returns the statevector from the trotterized Ising evolution of d qubits.
        """
        qc = self.get_trotterized_ising_circuit()
        qc.save_statevector()

        simulator = AerSimulator()
        result = simulator.run(transpile(qc, simulator)).result()
        return result.get_statevector()

def get_QPA_circuit(d, N, single_control=False,reduced_if=False):
    #FUNCTION TO GET QPA CIRCUIT
    if single_control:
       ncontrols=1
       get_control = lambda x: 0
    else:
       ncontrols=d
       get_control = lambda x: x
    cr_q0 = ClassicalRegister(ncontrols,name='control')
    qr_all = QuantumRegister(3*d+ncontrols)

    #Initialize quantum circuit with classical registers
    qcSWAP = QuantumCircuit(qr_all, cr_q0)  #Extra qubit and classical bit for parity check
    
    def recursive(N,qc):
      for k in range(ncontrols):
        qc.reset(k)
      qc.h(0) #q0_firstqbit = |0>+|1>/sqrt2
      for k in range(ncontrols-1):
        qc.cx(0,1+k) #q0 = |0000...> + |1111...>/sqrt2
      # Apply the first CSWAP gate controlled by q0, targeting q1 and q2
      for k in range(d):
        qc.cswap(get_control(k), k+ncontrols, k+d+ncontrols) #|+>_k x SYM12_k + |->_k x AntiSYM12_k /norm
      # Apply the second Hadamard gate to q0
      for k in range(ncontrols):
        qc.h(k) #|0> x SYM12 + |1> x AntiSYM12 /norm
      # Measure qubits 0 to d-1 into classical bits 0 to d-1
      
      # ================== measurement part ===================
      # for i in range(ncontrols): #Measure the control registers and find z
      #     qc.measure(i, cr_q0[i])
      #     if i==0:
      #       parity_control = qiskit_classical.expr.lift(cr_q0[i])
      #     else:
      #       parity_control = qiskit_classical.expr.bit_xor(parity_control, cr_q0[i])
      
      # ================== conditional part ===================
      # with qc.if_test(parity_control) as _else:
      #   #--------Z=1
      #   pass
      # with _else:
      #   #---------Z = 0
      #   # qc.x(d+1) # Good test to make sure it's working
      #   for k in range(d):
      #     qc.swap(k+d+ncontrols, k+2*d+ncontrols) #Swap q2 with q3
      #   if not reduced_if:
      #     if N!=1:
      #       qc = recursive(N-1,qc) #Do it again unless it was the final iteration of the SWAPNET
            
      # if reduced_if:
      #    if N!=1:
      #     qc = recursive(N-1,qc)
      return qc
    
    if N!=0:
      qc = recursive(N,qcSWAP)
    else:
      qc = qcSWAP
    # Gets Measure register q3 and save in the classical register
    # for i in range(d):
    #     qc.measure(3*d+i, cr_q0[i]) 
    qc.measure_all()
    return qc

def run_qc_and_return_state(qc):

    # Select Aer Simulator backend
    simulator = AerSimulator()

    def execute_circuit_on_state(qc):
        """ Executes a circuit on the AerSimulator and returns the state result. """
        qc_transpiled = transpile(qc, simulator)
        result = simulator.run(qc_transpiled).result()
        return result.get_statevector(qc_transpiled)

    qc.save_statevector()
    state = execute_circuit_on_state(qc)

    return state

def run_qc_and_return_result(qc, shots=1024000, sampler = SamplerV2(mode=fake_backend)):
    #RUN THE CIRCUIT AND MEASURE CLASSICALLY THE STATE IN REGISTER Q3, SHOULD RETURN A SEQUENCE OF BITS OF d DIMENSIONS FOR EACH SHOT, REPRESENTING THE FINAL STATE MEASURED
    """ Executes a circuit on the AerSimulator and returns the state result. """
    qc_transpiled = transpile(qc, backend=fake_backend, optimization_level=3)
    result = sampler.run([qc_transpiled],shots=shots).result()[0]
    return result

In [36]:
# Run the function
k=2
epsilon = 0.3
N_qpa = 1

QPA = get_QPA_circuit(k, N_qpa, True)
result = run_qc_and_return_result(QPA)
print(result.data.meas.get_counts())

print(QPA.draw())

{'0000000': 832079, '0000100': 33702, '0000001': 23975, '0001000': 22911, '0001010': 7197, '0001001': 6717, '0000011': 11588, '1000000': 4418, '0000010': 21574, '0010000': 11432, '0001011': 5405, '0000101': 8378, '0000111': 726, '0010100': 15740, '0001101': 535, '0010001': 3221, '0010101': 4279, '0100000': 2887, '0000110': 952, '0011101': 215, '0011011': 79, '0011001': 178, '0010010': 332, '0001111': 271, '0011110': 176, '0010011': 259, '0010110': 475, '0100001': 91, '0011100': 477, '0100010': 67, '1011101': 3, '0001110': 362, '0011111': 150, '0011000': 306, '0001100': 980, '0011010': 105, '1000100': 181, '0100100': 114, '1010101': 25, '1000010': 116, '1100000': 17, '0010111': 324, '1001000': 116, '0100101': 37, '1011001': 2, '0101000': 62, '1001011': 27, '1000001': 115, '0110000': 35, '1010000': 65, '1000111': 9, '1010001': 16, '0100011': 35, '1000011': 65, '1001010': 39, '0100110': 3, '1011100': 2, '0110100': 46, '1010100': 83, '0110101': 13, '0101001': 37, '1000101': 45, '0101010': 

In [ ]:
#DO THE PLOTTING ------------------------- To be added 
import numpy as np
import matplotlib.pyplot as plt
import time  # Import time module for tracking execution time
from tqdm import tqdm
list_of_lambda = [i * 0.05 for i in range(20)]
list_of_Nqpa = [0, 1, 2]
purified_fidelity = {i: [] for i in list_of_Nqpa}
theorical_fidelity=[]
k = 4
shots = 1024*10
debug = False # Set to False to disable print statements

for LAMBDA in tqdm(list_of_lambda, desc="Lambda Loop"):
    # Theoretical max fidelity
    if k==2:#1/8 (-2 + \[Lambda]) (1 + \[Lambda]) (-4 + 3 \[Lambda])
        theorical_fidelity.append(1/8 * (-2 + LAMBDA) * (1 + LAMBDA) * (-4 + 3*LAMBDA))
    elif k==3: #1/96 (-8 + 7 \[Lambda]) (-12 + 7 (-1 + \[Lambda]) \[Lambda])
        theorical_fidelity.append(1/96 * (-8 + 7*LAMBDA) * (-12 + 7*(-1 + LAMBDA)*LAMBDA))
    elif k==4:#1/128 (-16 + 15 \[Lambda]) (-8 + 5 (-1 + \[Lambda]) \[Lambda])
        theorical_fidelity.append(1/128 * (-16 + 15*LAMBDA) * (-8 + 5*(-1 + LAMBDA)*LAMBDA))
    else:#Just give 0 for now
        theorical_fidelity.append(0)
        

    if debug:
        print(f'Running: Epsilon={LAMBDA}')
        
    for Nqpa in list_of_Nqpa:
        if debug:
            print(f'Running: Epsilon={LAMBDA}, Nqpa={Nqpa}')
        start_time = time.time()

        QPA = get_QPA_circuit(k, LAMBDA, Nqpa)
        mid_time_qpa = time.time()
        if debug:
            print(f'For Epsilon={LAMBDA}, Nqpa={Nqpa}, found the QPA in {mid_time_qpa - start_time:.2f} seconds')

        results = run_qc_and_return_result(QPA, shots)
        mid_time_results = time.time()
        if debug:
            print(f'For Epsilon={LAMBDA}, Nqpa={Nqpa}, found the Results in {mid_time_results - mid_time_qpa:.2f} seconds')

        memory = results.get_memory()
        fidelity = memory.count('0' * k) / len(memory)
        purified_fidelity[Nqpa].append(fidelity)

        end_time = time.time()
        if debug:
            print(f'For Epsilon={LAMBDA}, Nqpa={Nqpa}, found the Fidelity in {end_time - mid_time_results:.2f} seconds')
            print(f'Total time for Epsilon={LAMBDA}, Nqpa={Nqpa}: {end_time - start_time:.2f} seconds\n')


# Plot results
for Nqpa in list_of_Nqpa:
    fidelity = np.array(purified_fidelity[Nqpa])
    if Nqpa==0:
        label = 'Base Fidelity'
    else:
        label = f'Fidelity after {Nqpa} Swapnets'
    plt.errorbar(list_of_lambda, purified_fidelity[Nqpa],
                 yerr=np.sqrt(fidelity/shots), label=label)

plt.plot(list_of_lambda, theorical_fidelity, label='Theoretical Maximum Fidelity')

plt.xlabel('Lambda')
plt.ylabel('Fidelity')
plt.title('Fidelity vs Lambda for Different Nqpa')
plt.legend()
plt.show()

Lambda Loop:   0%|          | 0/20 [00:00<?, ?it/s]


AttributeError: 'SamplerResult' object has no attribute 'get_memory'